# Synthetic complaint text per SNOMED reason code.
# Each entry maps a SNOMED code to symptom phrasings, optionally restricted by age or sex.

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Complaint templates

Kept separate from the build so the wording is easy to edit. A template may never
contain a word from its own diagnosis, so the text model cannot string-match the
label; `check_no_leakage` enforces that at build time.

In [ ]:
COMPLAINTS = {
    "431857002": {
        "any": [
            "Swollen ankles and puffiness around the eyes in the morning",
            "Persistent tiredness and itching all over",
            "Nausea and a metallic taste in the mouth",
            "Back pain and waking several times to pass water",
            "Poor appetite and unintentional weight loss",
        ],
        "older": [
            "Frail and lightheaded on standing, with swollen feet",
            "Appetite lost and very tired since my last review",
        ],
    },
    "46177005": {
        "any": [
            "Constant fatigue and breathlessness even at rest",
            "Itchy dry skin that flakes and a poor appetite",
            "A metallic taste in the mouth and no energy at all",
            "Heavy swelling in both legs by the evening",
            "Leg cramps and restlessness that keep me awake",
        ],
    },
    "72892002": {
        "any": [
            "Routine antenatal check and review of progress",
            "Feeling well and just attending the routine check",
            "Nausea in the first trimester and some tiredness",
            "Backache and difficulty sleeping in later months",
            "Booking the routine scan and blood tests",
        ],
    },
    "103697008": {
        "any": [
            "Sent by my doctor for a check-up",
            "Booked in for a scale and polish by my doctor",
            "Tooth sensitivity and bleeding when brushing",
            "Need a filling and a routine check",
            "Jaw ache and a broken tooth needing attention",
        ],
    },
    "66383009": {
        "any": [
            "Bleeding gums when I brush my teeth",
            "Gums look swollen and sore",
            "A bad taste in the mouth and bad breath",
            "Gums bleed a lot and feel tender when eating",
            "Swollen sore gums that started a few weeks ago",
        ],
    },
    "389095005": {
        "any": [
            "Review of my current birth control method",
            "Prescription renewal for my usual method",
            "Discussing hormone levels and family planning options",
            "Routine check and general health review",
            "Wanted to change to a different method",
        ],
        "female": [
            "Would like to discuss stopping my current method",
            "Family planning appointment and a repeat prescription",
        ],
    },
    "609328004": {
        "any": [
            "Sneezing and itchy watery eyes in the spring",
            "Wheezy chest and a runny nose after being outdoors",
            "A rash and swelling after eating certain foods",
            "Repeated sneezing and stuffiness all year round",
            "Watery itchy eyes and a scratchy throat",
        ],
    },
    "6525002": {
        "any": [
            "Trouble cutting down on a habit I want to break",
            "Family and work problems linked to substance use",
            "Craving and withdrawal symptoms affecting my sleep",
            "Struggling to stop using and feeling low about it",
            "Need help to cut back on a long-standing habit",
        ],
    },
    "88805009": {
        "any": [
            "Breathless lying flat and needing extra pillows",
            "Swollen ankles and weight gain over a few days",
            "Waking in the night short of breath",
            "Tiredness and puffiness in the face in the morning",
            "Reduced ability to walk without stopping",
        ],
    },
    "444814009": {
        "any": [
            "Blocked nose with facial pressure and thick mucus",
            "Facial pain that worsens when bending forward",
            "Headache around the eyes and a reduced sense of smell",
            "Runny nose and pressure behind the cheekbones",
            "Nasal congestion and a dull ache in the upper teeth",
        ],
    },
    "55822004": {
        "any": [
            "Cholesterol results follow-up",
            "Wanting advice on fat and cholesterol in my diet",
            "Family history of raised cholesterol and a blood test",
            "Review of cholesterol levels and lifestyle changes",
            "Lipid panel showing raised cholesterol",
        ],
    },
    "82423001": {
        "any": [
            "Aching in several joints that will not settle",
            "A long-standing lower back problem limiting my walking",
            "Constant dull aching and poor sleep",
            "A problem that has lasted for years and needs review",
            "Aching all over and difficulty getting up in the morning",
        ],
    },
    "254837009": {
        "any": [
            "A new lump and a change noticed in one area of the chest",
            "Follow-up appointment after a recent change noticed",
            "Nipple discharge and skin dimpling noticed recently",
            "Follow-up for an abnormality found on a recent scan",
            "A lump and tenderness in one area",
        ],
        "female": [
            "A new lump and some skin puckering in one area",
            "Discharge and a change in the shape noticed last week",
        ],
    },
    "195662009": {
        "any": [
            "Painful swallowing and a sore throat with a fever",
            "Throat pain and a runny nose that started two days ago",
            "A dry sore throat and mild shivers",
            "Difficulty swallowing liquids because of throat pain",
            "A raw throat and swollen glands in the neck",
        ],
    },
    "10509002": {
        "any": [
            "A cough with chest tightness that followed a cold",
            "A cough that keeps me awake and a mild wheeze",
            "A wheezy chest and a productive cough for two weeks",
            "A cough and breathlessness with no fever",
            "A deep cough and burning in the chest on breathing in",
        ],
    },
    "275978004": {
        "any": [
            "Routine stool sample check as advised by my doctor",
            "Booked for a routine check after turning fifty",
            "Routine appointment and results review",
            "Home test kit returned and the results discussed",
            "Test follow-up and the next steps",
        ],
    },
    "37320007": {
        "any": [
            "Missing molars and difficulty chewing",
            "Gaps in my mouth and a sore gum",
            "A broken tooth and pain when biting",
            "Loose molars and difficulty eating hard food",
            "My plate is loose and there is a gap at the back",
        ],
    },
    "1871000124103": {
        "any": [
            "Discharge planning and nursing visits after a hospital stay",
            "Setting up support to continue recovery after discharge",
            "Help needed after a recent hospital stay",
            "Nursing and a rehabilitation plan after discharge",
            "Arranging support and equipment after a hospital stay",
        ],
    },
    "414545008": {
        "any": [
            "Chest tightness when walking uphill",
            "Pressure in the chest and breathlessness on exertion",
            "Chest discomfort that settles after resting",
            "Ankle swelling and reduced exercise tolerance",
            "Palpitations and chest heaviness in the evening",
        ],
    },
    "424132000": {
        "any": [
            "Follow-up after an operation on a growth in my chest",
            "Imaging review and monitoring after a chest operation",
            "Shortness of breath and a dry cough as a smoker",
            "Routine review scans and blood tests",
            "Regaining strength and clearing a chest cough",
        ],
    },
    "36971009": {
        "any": [
            "A stuffy nose and an ache under one eye",
            "Pressure behind the nose and a dull head ache",
            "A blocked nose with yellow discharge",
            "Pressure behind the nose and difficulty smelling",
            "Facial heaviness and a cough that worsens at night",
        ],
    },
    "195967001": {
        "any": [
            "Wheezing and shortness of breath after exercise",
            "A night-time cough and chest tightness",
            "Using my inhaler more often than usual",
            "A cough and wheeze triggered by cold air",
            "Chest tightness that wakes me during the night",
        ],
    },
    "75498004": {
        "any": [
            "Facial pain with thick green discharge and a fever",
            "A blocked nose and facial pressure for over a week",
            "Greenish nasal discharge and pain when leaning forward",
            "Face pain and a fever with a blocked nose",
            "Purulent discharge and severe pressure behind one eye",
        ],
    },
    "271737000": {
        "any": [
            "Constant tiredness and looking pale",
            "Breathlessness on stairs with heavy fatigue",
            "Dizzy spells and unusually low energy",
            "Pale lips and feeling wiped out by midday",
            "Fatigue and palpitations with light-headedness",
        ],
    },
    "1149222004": {
        "any": [
            "Taken too many tablets and feeling drowsy",
            "Accidental extra dose of my medication",
            "A deliberate extra dose and feeling very unwell",
            "Swallowed a large amount of pills by mistake",
            "Drowsy and nauseous after a dosing error",
        ],
    },
    "59621000": {
        "any": [
            "Consistently high blood pressure readings at home",
            "Headaches and elevated readings at the pharmacy",
            "Regular blood pressure check and medication review",
            "Woke with a pounding head and a high reading",
            "Dizzy spells with high blood pressure measurements",
        ],
    },
    "192127007": {
        "any": [
            "Trouble focusing in class and difficulties with schoolwork",
            "Difficulty settling and interrupting constantly",
            "Forgetful and disorganised with school tasks",
            "Hyperactive at school and poor focus on homework",
            "Restless and impulsive with poor concentration",
        ],
    },
    "65363002": {
        "any": [
            "An earache and a feeling of pressure in the ear",
            "A blocked ear with reduced hearing and pain",
            "Night-time ear pain and a fever",
            "Ear discharge and pain when the head is turned",
            "Sharp ear pain and pressure inside",
        ],
    },
    "125605004": {
        "any": [
            "A broken wrist after a fall and a lot of swelling",
            "A swollen and bruised ankle after twisting it",
            "Wrist pain and deformity after slipping",
            "Hip pain and unable to bear weight after a fall",
            "Pain and swelling around a shoulder after a tumble",
        ],
    },
    "25675004": {
        "any": [
            "Moving into a residential setting and support with the move",
            "Rehabilitation placement and daily care needs",
            "A placement arrangement for ongoing daily support",
            "Placement in a residential setting for continued help",
            "Moving from hospital into long-term residential care",
        ],
    },
    "384709000": {
        "any": [
            "A twisted ankle with swelling and bruising",
            "A wrist twisted awkwardly and now very painful",
            "A swollen foot after rolling it on uneven ground",
            "Pain and bruising around a wrist joint after a twist",
            "An ankle ligament injury with difficulty putting weight on it",
        ],
    },
    "312608009": {
        "any": [
            "A deep cut on the arm that needed stitches",
            "A bleeding cut on the thigh after falling",
            "A gaping cut on the hand from broken glass",
            "A cut to the lower leg and bleeding heavily",
            "Stitching needed for a cut on the forearm",
        ],
    },
    "43878008": {
        "any": [
            "Painful white patches on the tonsils with a fever",
            "Sudden severe pain on swallowing with swollen glands",
            "A fever with a coated tongue and painful swallowing",
            "Painful swallowing of pills and glands in the neck",
            "Inflamed tonsils and sharp pain on swallowing",
        ],
    },
    "307426000": {
        "any": [
            "Burning when I pass water and a constant urge to go",
            "Lower belly pain and passing water every hour",
            "Cloudy strong-smelling water and burning",
            "Burning and blood when I pass water since yesterday",
            "Frequent painful urination with lower abdominal discomfort",
        ],
    },
    "233678006": {
        "any": [
            "A night-time cough and wheeze in the young",
            "Coughing fits that come on in the night",
            "Exercise-related wheeze and a persistent cough",
            "Wheeze and chest tightness after playing outside",
            "Repeated coughing at night and laboured breathing",
        ],
    },
    "183996000": {
        "any": [
            "Discussing a long-term method of preventing a pregnancy",
            "Looking into a permanent option after years of contraception",
            "Wanting to discuss a permanent family planning choice",
            "Family planning discussion about a permanent solution",
            "Hoping to stop using contraception for good",
        ],
        "female": [
            "Would like a permanent method discussed with me",
            "Long-term family planning options and a referral",
        ],
    },
    "161665007": {
        "any": [
            "Routine check after a previous organ operation",
            "Long-term follow-up after an earlier operation",
            "Blood tests and a medication review at my usual clinic",
            "Regular monitoring and a routine review",
            "A scheduled check and bloods at my usual clinic",
        ],
    },
    "156073000": {
        "any": [
            "Heavy bleeding and cramping after an early loss",
            "Follow-up after an early pregnancy loss",
            "Bleeding and abdominal pain after a loss",
            "A check-up after an early loss and feeling low",
            "A recovery review once the bleeding has settled",
        ],
    },
    "39898005": {
        "any": [
            "Waking repeatedly through the night and never feeling rested",
            "Difficulty settling at night and daytime tiredness",
            "Loud snoring and feeling exhausted during the day",
            "Trouble winding down at night and nodding off in the day",
            "Unrefreshing rest and nodding off in the afternoon",
        ],
    },
    "274531002": {
        "any": [
            "A review of an unexpected result on a recent scan",
            "Discussing an inconclusive result on a tracing",
            "A follow-up for a borderline reading on a test",
            "Concern about an unusual result on a routine scan",
            "A results review for a scan that came back unclear",
        ],
    },
}

FALLBACK_COMPLAINTS = {
    "emergency": [
        "Symptoms that started today and got worse fast",
        "Fainting at work and feeling very unwell",
        "Crushing chest discomfort radiating to my arm",
        "Breathless and collapsing, needed help straight away",
        "Weakness down one side of the body and difficulty speaking",
    ],
    "inpatient": [
        "Admitted and needing treatment on the ward",
        "Overnight stay and a planned procedure",
        "Admitted for observation and further tests",
    ],
    "outpatient": [
        "A new problem I have never been seen for",
        "A routine appointment and a review of my results",
        "A follow-up after my last appointment",
        "Advice about a problem that keeps recurring",
        "A check on an ongoing problem",
    ],
    "ambulatory": [
        "General follow-up and a repeat prescription",
        "A routine visit to discuss an ongoing problem",
        "A general check-up and medication review",
        "A results review and next steps",
    ],
    "virtual": [
        "Video appointment to discuss symptoms",
        "A remote consultation and a new prescription",
    ],
    "wellness": [
        "A wellness visit and a general check-up",
    ],
    "home": [
        "A doorstep visit to check on how I am coping",
    ],
    "snf": [
        "A progress review now that I live in a residential setting",
    ],
    "urgentcare": [
        "Attended an urgent treatment centre about a new problem",
        "A same-day appointment for something that flared up",
        "Walked in and was seen within a couple of hours",
        "Needed to be seen today rather than waiting for my usual doctor",
    ],
    "hospice": [
        "A palliative visit and support for my family",
        "A symptom control review and a chat about comfort",
        "A check-in visit and a discussion about the coming weeks",
    ],
}

## Build df

One row per encounter that carries a reason code: age and gender from
`patients.csv`, the complaint, and the SNOMED-CT reason code as the target.

In [ ]:
ARCHIVE = Path("archive")
CACHE = ARCHIVE / "df_ehr.csv"
MIN_CLASS_COUNT = 10
SEED = 42
ADULT_AGE = 18
OLDER_AGE = 65

SEMANTIC_TAGS = {
    "disorder", "finding", "procedure", "regime", "therapy", "situation",
    "event", "morphologic", "abnormality", "circulat",
}

STOPWORDS = {
    "of", "the", "a", "an", "and", "or", "in", "on", "to", "for", "with",
    "no", "not", "stage", "1", "2", "3", "4", "tnm", "non", "small",
    "screening", "history", "past", "family", "other", "unspecified",
    "finding", "disorder", "procedure", "regime", "therapy", "situation",
    "event", "morphologic", "abnormality", "circulat", "acute", "chronic",
    "normal", "suspected", "likely", "possible", "primary", "secondary",
    "left", "right", "bilateral", "upper", "lower", "multiple", "single",
    "female", "male", "at", "by", "from", "as", "is", "was", "has", "have",
    "my", "me", "i", "it", "they", "them", "he", "she", "his", "her",
    "this", "that", "these", "those", "been", "being", "am", "are", "do",
    "does", "did", "can", "could", "will", "would", "should", "there",
    "here", "up", "down", "out", "off", "over", "under", "than", "then",
    "into", "about", "after", "before", "between", "both", "each", "few",
    "more", "most", "own", "same", "so", "such", "too", "very", "just",
    "also", "now", "only", "why", "how", "all", "any",
}

# Bucket -> condition on the row that must hold for the variant to be usable.
BUCKET_FILTERS = {
    "any": None,
    "child": lambda age, sex: age < ADULT_AGE,
    "older": lambda age, sex: age >= OLDER_AGE,
    "female": lambda age, sex: sex == "Female",
    "male": lambda age, sex: sex == "Male",
}


def clean_description(raw):
    return re.sub(r"\s*\(([^)]*)\)\s*$", "", str(raw)).strip()


def stems(text):
    out = set()
    for word in re.findall(r"[a-z]+", text.lower()):
        if len(word) < 3 or word in STOPWORDS or word in SEMANTIC_TAGS:
            continue
        trimmed = re.sub(r"(ing|ed|es|s|ly)$", "", word)
        out.add(trimmed if len(trimmed) >= 4 else word)
    return out


def check_no_leakage(targets, per_code_templates, fallback_templates):
    collisions = []
    unused = sorted(set(per_code_templates) - set(targets))
    for code, variants in per_code_templates.items():
        if code not in targets:
            continue
        target_stems = stems(targets[code])
        for variant in variants:
            shared = stems(variant) & target_stems
            if shared:
                collisions.append((code, targets[code], sorted(shared), variant))
    fallback_variants = [v for group in fallback_templates.values() for v in group]
    for variant in fallback_variants:
        variant_stems = stems(variant)
        for code, target in targets.items():
            shared = variant_stems & stems(target)
            if shared:
                collisions.append((code, target, sorted(shared), variant))
    return collisions, unused


def flatten_templates(spec):
    variants, buckets = [], []
    for bucket, texts in spec.items():
        for text in texts:
            variants.append(text)
            buckets.append(bucket)
    return variants, buckets


def assign_complaints(codes, ages, sexes, encounter_classes):
    rng = np.random.default_rng(SEED)
    complaints = np.empty(len(codes), dtype=object)

    unknown = set(np.unique(encounter_classes)) - set(FALLBACK_COMPLAINTS)
    if unknown:
        raise KeyError(f"no fallback complaints for encounter classes {sorted(unknown)}")

    uniq, inverse = np.unique(codes, return_inverse=True)
    order = np.argsort(inverse, kind="stable")
    bounds = np.concatenate([[0], np.cumsum(np.bincount(inverse))])

    for position, code in enumerate(uniq):
        rows = order[bounds[position]:bounds[position + 1]]
        group_ages = ages[rows]
        group_sexes = sexes[rows]
        group_classes = encounter_classes[rows]

        if code in COMPLAINTS:
            variants, buckets = flatten_templates(COMPLAINTS[code])
            class_masks = None
        else:
            variants, classes = [], []
            for key, texts in FALLBACK_COMPLAINTS.items():
                for text in texts:
                    variants.append(text)
                    classes.append(key)
            buckets = ["any"] * len(variants)
            class_masks = [group_classes == key for key in classes]

        allowed = np.zeros((len(rows), len(variants)), dtype=bool)
        for column, bucket in enumerate(buckets):
            bucket_filter = BUCKET_FILTERS[bucket]
            allowed[:, column] = (
                np.ones(len(rows), dtype=bool) if bucket_filter is None
                else bucket_filter(group_ages, group_sexes)
            )
        if class_masks is not None:
            for column, mask in enumerate(class_masks):
                allowed[:, column] &= mask

        scores = rng.random(allowed.shape)
        scores[~allowed] = -1.0
        chosen = scores.argmax(axis=1)
        complaints[rows] = np.array(variants, dtype=object)[chosen]

    return complaints


TEXT_COLUMNS = [
    "Patient_ID", "Encounter_ID", "Gender", "Complaint",
    "Target_Diagnosis", "SNOMED_Code", "Encounter_Class",
]


def build_ehr_df():
    if CACHE.exists():
        return pd.read_csv(CACHE, dtype={c: str for c in TEXT_COLUMNS})

    started = time.time()
    patients = pd.read_csv(
        ARCHIVE / "patients.csv", usecols=["Id", "BIRTHDATE", "GENDER"]
    )
    encounters = pd.read_csv(
        ARCHIVE / "encounters.csv",
        usecols=[
            "Id", "START", "PATIENT", "ENCOUNTERCLASS", "DESCRIPTION",
            "REASONCODE", "REASONDESCRIPTION",
        ],
    )
    encounters = encounters[encounters["REASONCODE"].notna()].copy()
    encounters["START"] = pd.to_datetime(encounters["START"], utc=True)
    patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"], errors="coerce")
    print(f"loaded {len(encounters):,} encounters with a reason code")

    df = encounters.merge(patients, left_on="PATIENT", right_on="Id", how="inner")

    df["Age"] = df["START"].dt.year - df["BIRTHDATE"].dt.year
    had_birthday = (df["START"].dt.month > df["BIRTHDATE"].dt.month) | (
        (df["START"].dt.month == df["BIRTHDATE"].dt.month)
        & (df["START"].dt.day >= df["BIRTHDATE"].dt.day)
    )
    df["Age"] = df["Age"] - (~had_birthday).astype(int)
    df = df[df["Age"].between(0, 120)].copy()

    df["SNOMED_Code"] = df["REASONCODE"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["Target_Diagnosis"] = df["REASONDESCRIPTION"].map(clean_description)
    df["Encounter_Class"] = df["ENCOUNTERCLASS"].str.lower()
    df["Gender"] = df["GENDER"].map({"F": "Female", "M": "Male"})

    targets = df.groupby("SNOMED_Code")["Target_Diagnosis"].first().to_dict()
    per_code = {code: [v for texts in spec.values() for v in texts]
                for code, spec in COMPLAINTS.items()}
    collisions, unused = check_no_leakage(targets, per_code, FALLBACK_COMPLAINTS)
    if collisions:
        for code, target, shared, variant in collisions[:20]:
            print(f"LEAKAGE {code} [{target}] {shared} :: {variant}")
        raise ValueError(f"{len(collisions)} complaint templates leak the label")
    if unused:
        print(f"warning: templates for codes absent from the data: {unused}")
    print("complaint templates pass the leakage check")

    counts = df["SNOMED_Code"].value_counts()
    keep = counts[counts >= MIN_CLASS_COUNT].index
    df = df[df["SNOMED_Code"].isin(keep)].copy()
    print(f"dropped {len(counts) - len(keep)} classes with <{MIN_CLASS_COUNT} rows")

    df["Complaint"] = assign_complaints(
        df["SNOMED_Code"].to_numpy(),
        df["Age"].to_numpy(),
        df["Gender"].to_numpy(),
        df["Encounter_Class"].to_numpy(),
    )

    df = df.rename(columns={"PATIENT": "Patient_ID", "Id_x": "Encounter_ID"})
    df = df[[
        "Patient_ID", "Encounter_ID", "Age", "Gender", "Complaint",
        "Target_Diagnosis", "SNOMED_Code", "Encounter_Class",
    ]]
    df = df.dropna().reset_index(drop=True)
    df = df.astype({c: str for c in TEXT_COLUMNS})
    df.to_csv(CACHE, index=False)
    print(f"wrote {CACHE} with {len(df):,} rows in {time.time() - started:.0f}s")
    return df


df = build_ehr_df()

## Inspect df

In [ ]:
print(f"rows {len(df):,} | classes {df['SNOMED_Code'].nunique()} | patients {df['Patient_ID'].nunique()}")
print(df.dtypes.to_string())
print()
print(df["Target_Diagnosis"].value_counts().head(10).to_string())
print()
print(f"{df['Complaint'].nunique()} distinct complaint strings")
df.head(10)

## Model

Split by patient so no patient appears in both train and test, then cap each
class so the fit stays quick.

In [ ]:
MAX_PER_CLASS = 1000
N_SPLITS = 5

shuffled = df.sample(frac=1.0, random_state=SEED)
model_df = (
    shuffled.groupby("SNOMED_Code", sort=False)
    .head(MAX_PER_CLASS)
    .reset_index(drop=True)
)
print(
    f"modelling on {len(model_df):,} rows "
    f"across {model_df['SNOMED_Code'].nunique()} classes"
)

X = model_df[["Age", "Gender", "Complaint"]]
y = model_df["SNOMED_Code"]
groups = model_df["Patient_ID"]

splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
overlap = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
print(
    f"train {len(train_idx):,} rows / {groups.iloc[train_idx].nunique():,} patients\n"
    f"test  {len(test_idx):,} rows / {groups.iloc[test_idx].nunique():,} patients\n"
    f"patients in both splits: {overlap}"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("age", StandardScaler(), ["Age"]),
        ("gender", OneHotEncoder(handle_unknown="ignore"), ["Gender"]),
        (
            "complaint",
            TfidfVectorizer(max_features=2000, ngram_range=(1, 2)),
            "Complaint",
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=500, C=1.0)),
    ]
)

started = time.time()
model.fit(X_train, y_train)
print(
    f"fit in {time.time() - started:.0f}s "
    f"({model.named_steps['classifier'].n_iter_[0]} iterations)"
)

predictions = model.predict(X_test)
print(
    f"overall accuracy {accuracy_score(y_test, predictions):.3f} | "
    f"macro F1 {f1_score(y_test, predictions, average='macro', zero_division=0):.3f}"
)

is_templated = model_df["SNOMED_Code"].iloc[test_idx].isin(COMPLAINTS).to_numpy()
for label, mask in [
    ("diagnosis-specific complaint", is_templated),
    ("generic fallback complaint", ~is_templated),
]:
    if mask.any():
        print(
            f"{label}: {int(mask.sum()):,} test rows | "
            f"accuracy {accuracy_score(y_test[mask], predictions[mask]):.3f} | "
            f"macro F1 {f1_score(y_test[mask], predictions[mask], average='macro', zero_division=0):.3f}"
        )

print("\nreport on the diagnosis-specific subset:")
print(
    classification_report(
        y_test[is_templated],
        predictions[is_templated],
        zero_division=0,
        digits=3,
    )
)

new_patient = pd.DataFrame(
    [
        {
            "Age": 45,
            "Gender": "Female",
            "Complaint": "Wheezing and shortness of breath after exercise",
        }
    ]
)
predicted_code = model.predict(new_patient)[0]
print(f"predicted SNOMED_Code: {predicted_code}")
print(
    pd.Series(
        model.predict_proba(new_patient)[0],
        index=model.named_steps["classifier"].classes_,
    )
    .sort_values(ascending=False)
    .head(5)
)